<a href="https://colab.research.google.com/github/harshajs25/FLAPPY-BIRD--ESP-32/blob/main/road_ext.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!nvidia-smi
!pip install -q albumentations==1.3.1
print("✅ Ready")

Sat May  2 18:14:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             29W /   70W |     635MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/road_extraction'
os.makedirs(f'{WORK}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK}/predictions',  exist_ok=True)
os.makedirs(f'{WORK}/logs',         exist_ok=True)
print(f"✅ {WORK}")
# ==== CELL 2.5 — Copy dataset to fast local disk ====
import os
LOCAL_DATA = '/content/dataset'
os.makedirs(LOCAL_DATA, exist_ok=True)
print("Copying... (takes ~10 minutes, one time only)")
!cp -r /content/drive/MyDrive/deepglobe_road/train/* /content/dataset/
print(f"✅ Done. {len(os.listdir(LOCAL_DATA))} files copied.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ /content/drive/MyDrive/road_extraction
Copying... (takes ~10 minutes, one time only)
^C
✅ Done. 12458 files copied.


In [15]:
MODE   = "train"      # "train" or "predict"
RESUME = True        # True after a disconnect to continue training

DATASET_DIR = '/content/dataset'

IMG_SIZE      = 512   # random crop from 1024 source
BATCH_SIZE    = 16
NUM_EPOCHS    = 25
LR            = 2e-4
PATIENCE      = 8     # early stopping
NUM_WORKERS   = 2

BEST_CKPT = f'{WORK}/checkpoints/best_dlinknet34.pth'
LAST_CKPT = f'{WORK}/checkpoints/last_dlinknet34.pth'
LOG_CSV   = f'{WORK}/logs/training_log.csv'

# Predict-mode
INPUT_IMAGE    = '/content/drive/MyDrive/sample/sample_la.png'
OUTPUT_MASK    = f'{WORK}/predictions/road_mask.png'
OUTPUT_OVERLAY = f'{WORK}/predictions/road_overlay.png'

print(f"MODE={MODE}  RESUME={RESUME}")

MODE=train  RESUME=True


In [16]:
import numpy as np, cv2, csv, time
import torch, torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from glob import glob
import matplotlib.pyplot as plt
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: cuda


In [17]:
class DilatedCenter(nn.Module):
    """Cascaded dilated conv block — the 'D' in D-LinkNet.
    Captures multi-scale context essential for thin elongated roads."""
    def __init__(self, ch):
        super().__init__()
        self.d1 = nn.Conv2d(ch, ch, 3, padding=1,  dilation=1)
        self.d2 = nn.Conv2d(ch, ch, 3, padding=2,  dilation=2)
        self.d4 = nn.Conv2d(ch, ch, 3, padding=4,  dilation=4)
        self.d8 = nn.Conv2d(ch, ch, 3, padding=8,  dilation=8)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        a = self.relu(self.d1(x))
        b = self.relu(self.d2(a))
        c = self.relu(self.d4(b))
        d = self.relu(self.d8(c))
        return x + a + b + c + d


class DecoderBlock(nn.Module):
    """LinkNet decoder: 1x1 reduce → 3x3 transpose-conv upsample → 1x1 expand."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = in_ch // 4
        self.conv1   = nn.Conv2d(in_ch, mid, 1)
        self.bn1     = nn.BatchNorm2d(mid)
        self.deconv  = nn.ConvTranspose2d(mid, mid, 3, stride=2,
                                          padding=1, output_padding=1)
        self.bn2     = nn.BatchNorm2d(mid)
        self.conv2   = nn.Conv2d(mid, out_ch, 1)
        self.bn3     = nn.BatchNorm2d(out_ch)
        self.relu    = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.deconv(x)))
        x = self.relu(self.bn3(self.conv2(x)))
        return x


class DLinkNet34(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        r = tvm.resnet34(weights=tvm.ResNet34_Weights.IMAGENET1K_V1)
        self.first = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.e1, self.e2, self.e3, self.e4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.center = DilatedCenter(512)
        self.d4 = DecoderBlock(512, 256)
        self.d3 = DecoderBlock(256, 128)
        self.d2 = DecoderBlock(128,  64)
        self.d1 = DecoderBlock( 64,  64)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 3, padding=1),
        )
    def forward(self, x):
        x  = self.first(x)
        e1 = self.e1(x);  e2 = self.e2(e1)
        e3 = self.e3(e2); e4 = self.e4(e3)
        c  = self.center(e4)
        d4 = self.d4(c)  + e3
        d3 = self.d3(d4) + e2
        d2 = self.d2(d3) + e1
        d1 = self.d1(d2)
        return self.final(d1)

def build_model():
    return DLinkNet34().to(DEVICE)

In [18]:
class DeepGlobeRoads(Dataset):
    def __init__(self, imgs, masks, transform):
        self.imgs, self.masks, self.tf = imgs, masks, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        im = cv2.cvtColor(cv2.imread(self.imgs[i]), cv2.COLOR_BGR2RGB)
        mk = (cv2.imread(self.masks[i], cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        a = self.tf(image=im, mask=mk)
        return a['image'], a['mask'].unsqueeze(0)

def train_tf():
    return A.Compose([
        A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.4, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(0.0625, 0.2, 45, p=0.5, border_mode=cv2.BORDER_REFLECT),
        A.OneOf([
            A.RandomBrightnessContrast(0.2, 0.2, p=1.0),
            A.HueSaturationValue(15, 20, 15, p=1.0),
            A.CLAHE(p=1.0),
            A.RandomGamma((80, 120), p=1.0),
        ], p=0.7),
        A.OneOf([A.GaussNoise(p=1.0), A.GaussianBlur(p=1.0), A.MotionBlur(p=1.0)], p=0.2),
        A.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def val_tf():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

In [19]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def dice(self, p, t, eps=1e-7):
        p = torch.sigmoid(p)
        inter = (p * t).sum(dim=(1,2,3))
        denom = p.sum(dim=(1,2,3)) + t.sum(dim=(1,2,3))
        return 1 - ((2*inter + eps) / (denom + eps)).mean()
    def forward(self, logits, target):
        return 0.5 * self.dice(logits, target) + 0.5 * self.bce(logits, target)

def metrics(logits, target, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits) > thr).float()
    inter_r = (p * target).sum(dim=(1,2,3))
    union_r = p.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter_r
    riou = ((inter_r + eps) / (union_r + eps)).mean().item()
    inter_b = ((1-p) * (1-target)).sum(dim=(1,2,3))
    union_b = (1-p).sum(dim=(1,2,3)) + (1-target).sum(dim=(1,2,3)) - inter_b
    biou = ((inter_b + eps) / (union_b + eps)).mean().item()
    miou = (riou + biou) / 2
    f1 = 2 * riou / (1 + riou + eps)
    pacc = (p == target).float().mean().item()
    return riou, miou, f1, pacc

In [20]:
m = build_model()
n_params = sum(p.numel() for p in m.parameters()) / 1e6
print(f"D-LinkNet34: {n_params:.1f}M params")
del m; torch.cuda.empty_cache()

D-LinkNet34: 31.1M params


In [21]:
if MODE == "train":
    imgs  = sorted(glob(f'{DATASET_DIR}/*_sat.jpg'))
    masks = [p.replace('_sat.jpg', '_mask.png') for p in imgs]
    assert len(imgs) > 0, f"No images at {DATASET_DIR}"
    print(f"Found {len(imgs)} pairs")

    np.random.seed(42)
    idx = np.random.permutation(len(imgs))
    cut = int(0.9 * len(idx))
    ti, vi = idx[:cut], idx[cut:]

    tr_loader = DataLoader(
        DeepGlobeRoads([imgs[i] for i in ti], [masks[i] for i in ti], train_tf()),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    va_loader = DataLoader(
        DeepGlobeRoads([imgs[i] for i in vi], [masks[i] for i in vi], val_tf()),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    print(f"Train {len(ti)} | Val {len(vi)}")

    model     = build_model()
    criterion = DiceBCELoss()
    opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
    scaler    = torch.cuda.amp.GradScaler()

    start_epoch, best_iou, no_improve = 0, 0.0, 0

    if RESUME and os.path.exists(LAST_CKPT):
        ck = torch.load(LAST_CKPT, map_location=DEVICE)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched'])
        start_epoch = ck['epoch'] + 1
        best_iou    = ck['best']
        no_improve  = ck.get('no_improve', 0)
        print(f"⏯  Resuming epoch {start_epoch}, best={best_iou:.4f}")

    if not os.path.exists(LOG_CSV) or not RESUME:
        with open(LOG_CSV, 'w', newline='') as f:
            csv.writer(f).writerow(['epoch','time_min','train_loss','val_loss',
                                    'road_iou','mean_iou','f1','pixel_acc','lr'])

    for epoch in range(start_epoch, NUM_EPOCHS):
        t0 = time.time()
        model.train(); tloss = 0
        for im, mk in tqdm(tr_loader, desc=f'E{epoch+1}/{NUM_EPOCHS} train'):
            im, mk = im.to(DEVICE), mk.to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = criterion(model(im), mk)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tloss += loss.item()

        model.eval()
        vloss = riou = miou = f1 = pacc = 0
        with torch.no_grad():
            for im, mk in tqdm(va_loader, desc=f'E{epoch+1}/{NUM_EPOCHS} val'):
                im, mk = im.to(DEVICE), mk.to(DEVICE)
                with torch.cuda.amp.autocast():
                    lg = model(im)
                    vloss += criterion(lg, mk).item()
                r, m_, f, p_ = metrics(lg, mk)
                riou += r; miou += m_; f1 += f; pacc += p_

        n = len(va_loader)
        tloss /= len(tr_loader); vloss /= n
        riou /= n; miou /= n; f1 /= n; pacc /= n
        sched.step()
        dt = (time.time() - t0) / 60
        cur_lr = opt.param_groups[0]['lr']

        print(f"E{epoch+1:02d} [{dt:.1f}min] | train {tloss:.4f} | val {vloss:.4f} "
              f"| Road-IoU {riou:.4f} | mIoU {miou:.4f} | F1 {f1:.4f} | "
              f"PixAcc {pacc:.4f} | lr {cur_lr:.2e}")

        with open(LOG_CSV, 'a', newline='') as f:
            csv.writer(f).writerow([epoch+1, f'{dt:.1f}', tloss, vloss,
                                    riou, miou, f1, pacc, cur_lr])

        improved = riou > best_iou
        if improved:
            best_iou = riou
            no_improve = 0
        else:
            no_improve += 1

        ck = {'model': model.state_dict(), 'opt': opt.state_dict(),
              'sched': sched.state_dict(), 'epoch': epoch,
              'best': best_iou, 'no_improve': no_improve}
        torch.save(ck, LAST_CKPT)
        if improved:
            torch.save(ck, BEST_CKPT)
            print(f"   ✅ new best Road-IoU={best_iou:.4f}")

        if no_improve >= PATIENCE:
            print(f"⛔ Early stop — no improvement for {PATIENCE} epochs")
            break

    print(f"\n🏁 Done. Best Road-IoU = {best_iou:.4f}")

Found 6226 pairs
Train 5603 | Val 623


/tmp/ipykernel_3188/1266703757.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


⏯  Resuming epoch 25, best=0.5799

🏁 Done. Best Road-IoU = 0.5799


In [22]:
if MODE == "predict":
    assert os.path.exists(BEST_CKPT), "Train first."
    assert os.path.exists(INPUT_IMAGE), f"Missing: {INPUT_IMAGE}"

    model = build_model()
    ck = torch.load(BEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ck['model']); model.eval()
    print(f"Loaded model — best Road-IoU at save: {ck['best']:.4f}")

    img_bgr = cv2.imread(INPUT_IMAGE)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    print(f"Image: {W}x{H}")

    PATCH, STRIDE = IMG_SIZE, IMG_SIZE // 2
    pad_h = max(0, (PATCH - H % STRIDE) % STRIDE if H > PATCH else PATCH - H)
    pad_w = max(0, (PATCH - W % STRIDE) % STRIDE if W > PATCH else PATCH - W)
    img_p = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
    Hp, Wp = img_p.shape[:2]

    hann = np.outer(np.hanning(PATCH), np.hanning(PATCH)).astype(np.float32) + 1e-3
    prob = np.zeros((Hp, Wp), dtype=np.float32)
    wt   = np.zeros((Hp, Wp), dtype=np.float32)
    norm = A.Compose([A.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
                      ToTensorV2()])

    ys = sorted(set(list(range(0, Hp - PATCH + 1, STRIDE)) + [Hp - PATCH]))
    xs = sorted(set(list(range(0, Wp - PATCH + 1, STRIDE)) + [Wp - PATCH]))

    print(f"Patches: {len(ys) * len(xs)} × 8 TTA")
    with torch.no_grad():
        for y in tqdm(ys):
            for x in xs:
                t = norm(image=img_p[y:y+PATCH, x:x+PATCH])['image']\
                    .unsqueeze(0).to(DEVICE)
                preds = []
                # 4 rotations × 2 flips = 8 TTA
                for k in range(4):
                    for flip in [False, True]:
                        a = torch.rot90(t, k=k, dims=(2,3))
                        if flip: a = torch.flip(a, dims=[3])
                        with torch.cuda.amp.autocast():
                            o = torch.sigmoid(model(a))
                        if flip: o = torch.flip(o, dims=[3])
                        o = torch.rot90(o, k=-k, dims=(2,3))
                        preds.append(o)
                p = torch.stack(preds).mean(0).squeeze().cpu().numpy()
                prob[y:y+PATCH, x:x+PATCH] += p * hann
                wt[y:y+PATCH, x:x+PATCH] += hann

    prob = (prob / wt)[:H, :W]
    binary = (prob > 0.5).astype(np.uint8) * 255


In [24]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
cleaned = np.zeros_like(binary)
for i in range(1, n_labels):
  if stats[i, cv2.CC_STAT_AREA] >= 200:
    cleaned[labels == i] = 255
binary = cleaned

cv2.imwrite(OUTPUT_MASK, binary)

overlay = img_bgr.copy()
red = np.zeros_like(overlay); red[:,:,2] = 255
m3 = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR) > 0
overlay = np.where(m3, cv2.addWeighted(overlay, 0.5, red, 0.5, 0), overlay)
cv2.imwrite(OUTPUT_OVERLAY, overlay)

fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].imshow(img_rgb);             ax[0].set_title('Input')
ax[1].imshow(binary, cmap='gray'); ax[1].set_title('Predicted Roads')
ax[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); ax[2].set_title('Overlay')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

print(f"✅ Mask:    {OUTPUT_MASK}")
print(f"✅ Overlay: {OUTPUT_OVERLAY}")

NameError: name 'binary' is not defined

In [25]:
import os
ckpt = '/content/drive/MyDrive/road_extraction/checkpoints/best_dlinknet34.pth'
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"✅ Model saved: {size_mb:.1f} MB")
else:
    print("❌ Not found — something went wrong")

✅ Model saved: 373.5 MB


In [ ]:
# ==== CELL 11 — BATCH PREDICT (entire folder) ====

INPUT_FOLDER  = '/content/drive/MyDrive/sample'
OUTPUT_FOLDER = f'{WORK}/predictions/batch'
THRESHOLD     = 0.40   # use the best from threshold tuning, or 0.5 default

import os, glob, cv2, numpy as np, torch
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

os.makedirs(f'{OUTPUT_FOLDER}/masks',    exist_ok=True)
os.makedirs(f'{OUTPUT_FOLDER}/overlays', exist_ok=True)

# Load model once
assert os.path.exists(BEST_CKPT), "Train first."
model = build_model()
ck = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ck['model'])
model.eval()
print(f"✅ Model loaded — best Road-IoU: {ck['best']:.4f}")

# Find all images (jpg/jpeg/png/tif)
exts = ['*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff',
        '*.JPG', '*.JPEG', '*.PNG']
files = []
for e in exts:
    files.extend(glob.glob(os.path.join(INPUT_FOLDER, e)))
files = sorted(set(files))
print(f"Found {len(files)} images in {INPUT_FOLDER}")
assert len(files) > 0, f"No images found in {INPUT_FOLDER}"

# Reusable inference function
norm = A.Compose([A.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
                  ToTensorV2()])

def predict_image(img_path, use_tta=True):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: return None, None, None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]

    PATCH, STRIDE = IMG_SIZE, IMG_SIZE // 2
    pad_h = max(0, (PATCH - H % STRIDE) % STRIDE if H > PATCH else PATCH - H)
    pad_w = max(0, (PATCH - W % STRIDE) % STRIDE if W > PATCH else PATCH - W)
    img_p = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
    Hp, Wp = img_p.shape[:2]

    hann = np.outer(np.hanning(PATCH), np.hanning(PATCH)).astype(np.float32) + 1e-3
    prob = np.zeros((Hp, Wp), dtype=np.float32)
    wt   = np.zeros((Hp, Wp), dtype=np.float32)

    ys = sorted(set(list(range(0, Hp - PATCH + 1, STRIDE)) + [Hp - PATCH]))
    xs = sorted(set(list(range(0, Wp - PATCH + 1, STRIDE)) + [Wp - PATCH]))

    n_tta = 8 if use_tta else 1
    with torch.no_grad():
        for y in ys:
            for x in xs:
                t = norm(image=img_p[y:y+PATCH, x:x+PATCH])['image']\
                    .unsqueeze(0).to(DEVICE)
                preds = []
                if use_tta:
                    for k in range(4):
                        for flip in [False, True]:
                            a = torch.rot90(t, k=k, dims=(2,3))
                            if flip: a = torch.flip(a, dims=[3])
                            with torch.cuda.amp.autocast():
                                o = torch.sigmoid(model(a))
                            if flip: o = torch.flip(o, dims=[3])
                            o = torch.rot90(o, k=-k, dims=(2,3))
                            preds.append(o)
                else:
                    with torch.cuda.amp.autocast():
                        preds.append(torch.sigmoid(model(t)))

                p = torch.stack(preds).mean(0).squeeze().cpu().numpy()
                prob[y:y+PATCH, x:x+PATCH] += p * hann
                wt[y:y+PATCH, x:x+PATCH] += hann

    prob = (prob / wt)[:H, :W]
    binary = (prob > THRESHOLD).astype(np.uint8) * 255

    # Post-process: close gaps + remove tiny components
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
    cleaned = np.zeros_like(binary)
    for i in range(1, n_labels):
        if stats[i, cv2.CC_STAT_AREA] >= 200:
            cleaned[labels == i] = 255
    binary = cleaned

    overlay = img_bgr.copy()
    red = np.zeros_like(overlay); red[:,:,2] = 255
    m3 = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR) > 0
    overlay = np.where(m3, cv2.addWeighted(overlay, 0.5, red, 0.5, 0), overlay)
    return img_rgb, binary, overlay

# ---- Process all files ----
USE_TTA = True   # set False for ~8x faster but slightly lower quality
print(f"\nProcessing {len(files)} images (TTA={'ON' if USE_TTA else 'OFF'})...")

results = []
for fpath in tqdm(files, desc='predicting'):
    name = os.path.splitext(os.path.basename(fpath))[0]
    img_rgb, mask, overlay = predict_image(fpath, use_tta=USE_TTA)
    if mask is None:
        print(f"  ⚠️ Skipped (couldn't read): {fpath}")
        continue

    cv2.imwrite(f'{OUTPUT_FOLDER}/masks/{name}_mask.png', mask)
    cv2.imwrite(f'{OUTPUT_FOLDER}/overlays/{name}_overlay.png', overlay)
    road_pct = (mask > 0).mean() * 100
    results.append((name, road_pct))

print(f"\n✅ Done. Saved to:")
print(f"   Masks:    {OUTPUT_FOLDER}/masks/")
print(f"   Overlays: {OUTPUT_FOLDER}/overlays/")

# Save summary CSV
import csv
with open(f'{OUTPUT_FOLDER}/summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['filename', 'road_pixel_percent'])
    for name, pct in results:
        w.writerow([name, f'{pct:.2f}'])
print(f"   Summary:  {OUTPUT_FOLDER}/summary.csv")

# Show first 4 results as preview
n_show = min(4, len(results))
fig, axes = plt.subplots(n_show, 3, figsize=(18, 5*n_show))
if n_show == 1: axes = axes.reshape(1, -1)
for i, (name, _) in enumerate(results[:n_show]):
    img = cv2.cvtColor(cv2.imread(files[i]), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(f'{OUTPUT_FOLDER}/masks/{name}_mask.png', 0)
    ov  = cv2.cvtColor(cv2.imread(f'{OUTPUT_FOLDER}/overlays/{name}_overlay.png'),
                       cv2.COLOR_BGR2RGB)
    axes[i,0].imshow(img);             axes[i,0].set_title(f'{name} — input')
    axes[i,1].imshow(msk, cmap='gray'); axes[i,1].set_title('roads')
    axes[i,2].imshow(ov);              axes[i,2].set_title('overlay')
    for a in axes[i]: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(LOG_CSV)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(df['epoch'], df['train_loss'], label='Train loss')
ax[0].plot(df['epoch'], df['val_loss'],   label='Val loss')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title('Training & Validation Loss')

ax[1].plot(df['epoch'], df['road_iou'], label='Road IoU', color='C2')
ax[1].plot(df['epoch'], df['mean_iou'], label='Mean IoU', color='C3')
ax[1].plot(df['epoch'], df['f1'],       label='F1',       color='C4')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Score'); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title('Validation Metrics')

plt.tight_layout()
plt.savefig(f'{WORK}/logs/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved to {WORK}/logs/training_curves.png")

In [ ]:
# ==== CELL — Post-process existing prediction masks ====
import os, glob, cv2, numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

INPUT_MASK_FOLDER  = f'{WORK}/predictions/batch/masks'
INPUT_IMAGE_FOLDER = '/content/drive/MyDrive/road_extraction/sample'   # original images
OUTPUT_FOLDER      = f'{WORK}/predictions/batch_postprocessed'

os.makedirs(f'{OUTPUT_FOLDER}/masks',    exist_ok=True)
os.makedirs(f'{OUTPUT_FOLDER}/overlays', exist_ok=True)


def aggressive_post_process(binary_mask):
    """Stronger post-processing for cleaner road extraction."""
    # 1. Larger close kernel — fills bigger gaps in roads
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    out = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, k_close)

    # 2. Shape-aware component filtering
    #    Roads = elongated. Parking lots/buildings = blob-shaped.
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(out, 8)
    cleaned = np.zeros_like(out)
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w    = stats[i, cv2.CC_STAT_WIDTH]
        h    = stats[i, cv2.CC_STAT_HEIGHT]
        aspect = max(w, h) / max(min(w, h), 1)

        # Keep big components, OR medium ones that are road-shaped (elongated)
        if area >= 500:
            cleaned[labels == i] = 255
        elif area >= 150 and aspect >= 3:
            cleaned[labels == i] = 255

    # 3. Slight dilate to reconnect near-touching road segments
    k_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.dilate(cleaned, k_dilate, iterations=1)

    # 4. Final smoothing close
    k_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, k_smooth)

    return cleaned


# Find all existing masks
mask_files = sorted(glob.glob(f'{INPUT_MASK_FOLDER}/*_mask.png'))
print(f"Found {len(mask_files)} existing masks to post-process")
assert len(mask_files) > 0, f"No masks in {INPUT_MASK_FOLDER}"

for mpath in tqdm(mask_files, desc='post-processing'):
    name = os.path.basename(mpath).replace('_mask.png', '')

    # Load existing mask
    mask = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)

    # Apply stronger post-processing
    improved = aggressive_post_process(mask)

    # Save improved mask
    cv2.imwrite(f'{OUTPUT_FOLDER}/masks/{name}_mask.png', improved)

    # Make new overlay using ORIGINAL image
    img_paths = (glob.glob(f'{INPUT_IMAGE_FOLDER}/{name}.*'))
    if img_paths:
        img_bgr = cv2.imread(img_paths[0])
        if img_bgr is not None:
            # Resize mask to image size if needed
            if img_bgr.shape[:2] != improved.shape:
                improved_resized = cv2.resize(improved, (img_bgr.shape[1], img_bgr.shape[0]),
                                              interpolation=cv2.INTER_NEAREST)
            else:
                improved_resized = improved

            overlay = img_bgr.copy()
            red = np.zeros_like(overlay); red[:, :, 2] = 255
            m3 = cv2.cvtColor(improved_resized, cv2.COLOR_GRAY2BGR) > 0
            overlay = np.where(m3,
                               cv2.addWeighted(overlay, 0.5, red, 0.5, 0),
                               overlay)
            cv2.imwrite(f'{OUTPUT_FOLDER}/overlays/{name}_overlay.png', overlay)

print(f"\n✅ Done. Improved masks saved to:")
print(f"   {OUTPUT_FOLDER}/masks/")
print(f"   {OUTPUT_FOLDER}/overlays/")


# ---- Show before/after comparison for first 3 images ----
print("\nShowing before/after comparisons...")
n_show = min(3, len(mask_files))
fig, axes = plt.subplots(n_show, 3, figsize=(18, 5 * n_show))
if n_show == 1: axes = axes.reshape(1, -1)

for i in range(n_show):
    name = os.path.basename(mask_files[i]).replace('_mask.png', '')

    img_paths = glob.glob(f'{INPUT_IMAGE_FOLDER}/{name}.*')
    img = cv2.cvtColor(cv2.imread(img_paths[0]), cv2.COLOR_BGR2RGB) if img_paths else None
    before = cv2.imread(mask_files[i], 0)
    after  = cv2.imread(f'{OUTPUT_FOLDER}/masks/{name}_mask.png', 0)

    if img is not None:
        axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{name} — original')
    axes[i, 1].imshow(before, cmap='gray'); axes[i, 1].set_title('BEFORE post-process')
    axes[i, 2].imshow(after,  cmap='gray'); axes[i, 2].set_title('AFTER post-process')
    for a in axes[i]: a.axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_FOLDER}/comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [26]:
# ==== CELL — Threshold tuning ====
import torch, numpy as np
from tqdm import tqdm

# Load best model
model = build_model()
ck = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ck['model'])
model.eval()
print(f"Loaded model — saved Road-IoU: {ck['best']:.4f}")

# Collect raw probabilities on validation set
all_probs, all_masks = [], []
with torch.no_grad():
    for im, mk in tqdm(va_loader, desc='predicting val set'):
        im = im.to(DEVICE)
        with torch.cuda.amp.autocast():
            p = torch.sigmoid(model(im)).cpu().numpy()
        all_probs.append(p)
        all_masks.append(mk.numpy())
all_probs = np.concatenate(all_probs)
all_masks = np.concatenate(all_masks)

# Test thresholds
print(f"\n{'thr':<6} {'Road-IoU':<10} {'gain vs 0.5'}")
print("-" * 32)

baseline = None
best_thr, best_iou = 0.5, 0
for thr in [0.25, 0.30, 0.35, 0.40, 0.42, 0.45, 0.48, 0.50, 0.55, 0.60]:
    pred = (all_probs > thr).astype(np.float32)
    inter = (pred * all_masks).sum(axis=(1,2,3))
    union = pred.sum(axis=(1,2,3)) + all_masks.sum(axis=(1,2,3)) - inter
    iou = ((inter + 1e-7) / (union + 1e-7)).mean()
    if thr == 0.50: baseline = iou
    if iou > best_iou:
        best_iou, best_thr = iou, thr

    gain_str = f"+{iou - (baseline or iou):.4f}" if baseline else ""
    marker = "  ⭐" if thr == best_thr else ""
    print(f"{thr:<6.2f} {iou:<10.4f} {gain_str}{marker}")

print(f"\n✅ BEST threshold: {best_thr}")
print(f"   Road-IoU: {best_iou:.4f}  (vs 0.5 default: +{best_iou - baseline:.4f})")
print(f"\n💡 Use THRESHOLD = {best_thr} in your batch predict cell")

# Save it
import json
with open(f'{WORK}/checkpoints/best_threshold.json', 'w') as f:
    json.dump({'threshold': float(best_thr), 'iou': float(best_iou)}, f)

Loaded model — saved Road-IoU: 0.5799


predicting val set:   0%|          | 0/39 [00:00<?, ?it/s]/tmp/ipykernel_3188/3491607627.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
predicting val set: 100%|██████████| 39/39 [00:24<00:00,  1.59it/s]



thr    Road-IoU   gain vs 0.5
--------------------------------
0.25   0.5732       ⭐
0.30   0.5757       ⭐
0.35   0.5774       ⭐
0.40   0.5786       ⭐
0.42   0.5790       ⭐
0.45   0.5795       ⭐
0.48   0.5798       ⭐
0.50   0.5799     +0.0000  ⭐
0.55   0.5800     +0.0001  ⭐
0.60   0.5797     +-0.0002

✅ BEST threshold: 0.55
   Road-IoU: 0.5800  (vs 0.5 default: +0.0001)

💡 Use THRESHOLD = 0.55 in your batch predict cell
